# 02 — Baselines & Evaluation Pipeline

**Inputs:** `train/val/test.parquet` + `id_mappings.json` from notebook 01.

This notebook builds two things every later notebook depends on:

1. **The evaluation pipeline** — Recall@20/50, NDCG@10, Catalog Coverage, computed the exact same way for every model (baselines, MF-BPR, two-tower, LightGBM). One shared implementation = comparable numbers.
2. **The two required baselines** — Random and Popularity. These set the floor: if a learned model can't beat popularity, it's not learning anything useful.

It also **exports popularity scores** — the first artifact the webapp can serve (logged-out homepage + cold-user fallback).

## 0. Setup

In [1]:
import os, json
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/recsys_kindle/data'
except ImportError:
    DATA_DIR = './data'

train = pd.read_parquet(os.path.join(DATA_DIR, 'train.parquet'))
val   = pd.read_parquet(os.path.join(DATA_DIR, 'val.parquet'))
test  = pd.read_parquet(os.path.join(DATA_DIR, 'test.parquet'))
with open(os.path.join(DATA_DIR, 'id_mappings.json')) as f:
    mappings = json.load(f)

N_ITEMS = len(mappings['item2idx'])
N_USERS = len(mappings['user2idx'])
print(f'train {len(train):,} | val {len(val):,} | test {len(test):,} | vocab: {N_USERS:,} users, {N_ITEMS:,} items')

train 2,254,643 | val 281,830 | test 281,831 | vocab: 127,585 users, 120,573 items


## 1. Evaluation protocol

Decisions that apply to **every** model in this project (worth being explicit — these are classic defense questions):

- **Ground truth** = *positive* (rating ≥ 4) test-period interactions on items known at training time. We can't credit a model for recommending an item that didn't exist in its training catalog.
- **Evaluated users** = test users that exist in the training vocabulary. Users unseen at training time can't get personalized recs from CF models — in the webapp they get the popularity fallback; in `ANALYSIS.md` cold-start section we analyze the *low-activity* end of known users.
- **Seen-item filtering**: we exclude items the user already interacted with in training from their recommendations. Re-recommending an already-read book is pointless in this domain and would inflate metrics.
- **Metrics**: Recall@20, Recall@50 (retrieval quality at candidate-list size), NDCG@10 (ranking quality where position matters), Catalog Coverage@10 (what fraction of the catalog ever gets shown — diversity/health check).

In [2]:
def build_truth(part):
    """{user_idx: set(positive item_idx)} restricted to train vocabulary."""
    p = part[part.is_positive & part.user_idx.notna() & part.item_idx.notna()]
    return p.groupby('user_idx')['item_idx'].agg(set).to_dict()

truth_val  = build_truth(val)
truth_test = build_truth(test)

# Items each user saw during training (to exclude from recommendations)
seen_train = train.groupby('user_idx')['item_idx'].agg(set).to_dict()

print(f'Evaluable users: val {len(truth_val):,} | test {len(truth_test):,}')
avg_rel = np.mean([len(s) for s in truth_test.values()])
print(f'Avg positive test items per evaluable user: {avg_rel:.2f}')

Evaluable users: val 45,255 | test 34,191
Avg positive test items per evaluable user: 2.53


In [3]:
# Shared metrics (identical to src/metrics.py in the repo, inlined so the notebook is self-contained)
def recall_at_k(recs, truth, k):
    s = [len(set(recs.get(u, [])[:k]) & rel) / len(rel) for u, rel in truth.items() if rel]
    return float(np.mean(s)) if s else 0.0

def ndcg_at_k(recs, truth, k):
    scores = []
    for u, rel in truth.items():
        if not rel:
            continue
        top = recs.get(u, [])[:k]
        dcg = sum(1.0 / np.log2(i + 2) for i, it in enumerate(top) if it in rel)
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(rel), k)))
        scores.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(scores))

def catalog_coverage(recs, n_items, k):
    seen = set()
    for r in recs.values():
        seen.update(r[:k])
    return len(seen) / n_items

def evaluate(recs, truth, n_items, name):
    row = {'Model': name,
           'Recall@20': recall_at_k(recs, truth, 20),
           'Recall@50': recall_at_k(recs, truth, 50),
           'NDCG@10':   ndcg_at_k(recs, truth, 10),
           'Coverage@10': catalog_coverage(recs, n_items, 10)}
    return row

## 2. Random baseline

Uniformly random items per user (excluding train-seen). Expected to be near zero on recall/NDCG with a catalog this size — its only job is to confirm the pipeline works and to maximize coverage (it recommends everything eventually).

In [4]:
K_MAX = 50

def random_recommender(users):
    recs = {}
    for u in users:
        seen = seen_train.get(u, set())
        cand = rng.choice(N_ITEMS, size=K_MAX + len(seen), replace=False)
        recs[u] = [int(i) for i in cand if i not in seen][:K_MAX]
    return recs

recs_random = random_recommender(truth_test.keys())
row_random = evaluate(recs_random, truth_test, N_ITEMS, 'Random')
row_random

{'Model': 'Random',
 'Recall@20': 0.0001463997998173658,
 'Recall@50': 0.00042144251111985007,
 'NDCG@10': 3.5021441196450924e-05,
 'Coverage@10': 0.9414877294253274}

## 3. Popularity baseline

**Popularity = number of *positive* interactions in the training set** (not the full 27-year history — only what the model era would know). Every user gets the same head-of-catalog list, minus items they've already seen. This is the serious baseline: with a long-tail catalog, the head captures a big share of test interactions.

In [5]:
pop_counts = (train[train.is_positive]
              .groupby('item_idx').size()
              .sort_values(ascending=False))
pop_ranked = pop_counts.index.to_numpy().astype(int)
print('Top-5 most popular item_idx:', pop_ranked[:5], '\ncounts:', pop_counts.head().to_numpy())

def popularity_recommender(users):
    recs = {}
    head = pop_ranked[:K_MAX + 200]   # enough buffer to survive seen-filtering
    for u in users:
        seen = seen_train.get(u, set())
        recs[u] = [int(i) for i in head if i not in seen][:K_MAX]
    return recs

recs_pop = popularity_recommender(truth_test.keys())
row_pop = evaluate(recs_pop, truth_test, N_ITEMS, 'Popularity')
row_pop

Top-5 most popular item_idx: [92330 92743  1170 36776 44712] 
counts: [1412 1285 1170 1132 1050]


{'Model': 'Popularity',
 'Recall@20': 0.01406881885385847,
 'Recall@50': 0.024168574110404867,
 'NDCG@10': 0.005305601277063252,
 'Coverage@10': 0.0001575808846093238}

## 4. Results

In [6]:
results = pd.DataFrame([row_random, row_pop]).set_index('Model')
results.round(4)

,Recall@20,Recall@50,NDCG@10,Coverage@10
Model,,,,
Random,0.0001,0.0004,0.0000,0.9415
Popularity,0.0141,0.0242,0.0053,0.0002


**Interpretation.**

- *Random* — Recall@20 = 0.0001, Recall@50 = 0.0004, NDCG@10 ≈ 0.00004, Coverage@10 = 0.94. Confirms the pipeline works: with ~120K items, hitting a user's handful of positives by chance is negligible, and coverage is near-maximal by construction (it eventually recommends everything).
- *Popularity* — Recall@20 = 0.0141, Recall@50 = 0.0242, NDCG@10 = 0.0053, but Coverage@10 = 0.0002 (same ~10 books to nearly everyone). This is the number to beat: decent recall from the long-tail head effect, terrible diversity — exactly the argument for personalized retrieval in the next notebooks.
- These same functions (`evaluate`, `build_truth`, seen-filtering) are reused verbatim for MF-BPR, two-tower and LightGBM, so all rows of the final comparison table are apples-to-apples.

## 5. Export artifacts for the webapp

Popularity scores are a real serving artifact: the logged-out homepage and the cold-user fallback both use them. We export them now so the webapp can already run end-to-end with baseline recommendations (professor's advice: have something working early).

In [7]:
pop_df = pop_counts.rename('score').reset_index()
pop_df['score'] = pop_df['score'] / pop_df['score'].max()   # normalize for temperature sampling in the API
pop_df.to_parquet(os.path.join(DATA_DIR, 'popularity.parquet'), index=False)

with open(os.path.join(DATA_DIR, 'baseline_results.json'), 'w') as f:
    json.dump(results.reset_index().to_dict(orient='records'), f, indent=2)

print('Saved popularity.parquet + baseline_results.json to', DATA_DIR)

Saved popularity.parquet + baseline_results.json to ./data


## Next steps

- **03_mf_bpr** — MF with BPR loss from scratch (PyTorch), popularity-based negative sampling; tune on `val`, report on `test` with this pipeline.
- **04_two_tower** — two-tower retrieval + FAISS index; export item embeddings (also powers the webapp's "Similar items" and "Because you liked X").
- **05_ranking** — feature engineering + LightGBM LambdaRank re-ranking.
- Webapp population: `popularity.parquet` → Postgres `popularity` table (script in `app/`).